# Core

> bioMONAI core functions


In [ ]:
#| default_exp core

In [ ]:
#| hide
from nbdev.showdoc import *

## Imports

Main imports from other libraries.

In [ ]:
#| export
from torch import Tensor as torchTensor
from torch import tensor
from monai.data import MetaTensor
from monai.utils import set_determinism

import torch.nn.functional as F

from random import randint

from skimage import util
from skimage.data import cells3d

In [ ]:
#| export
from torch import squeeze as torchsqueeze, max as torchmax, from_numpy as torch_from_numpy, device as torch_device
from torch.cuda import is_available as is_cuda_available

In [ ]:
#| export
from collections.abc import MutableSequence
from typing import MutableSequence
    
from fastai.callback.core import Callback
from fastai.data.all import DataLoaders, Path, trainable_params, delegates, hasattrs, Path, List, L, typedispatch
from fastai.optimizer import Adam, OptimWrapper, Optimizer
from fastai.vision.all import BypassNewMeta, DisplayedTransform, store_attr, DataBlock, Learner, ShowGraphCallback, CSVLogger, Any, minimum, steep, valley, slide


In [ ]:
show_doc(DataBlock)


---

[source](https://github.com/fastai/fastai/blob/master/fastai/data/block.py#LNone){target="_blank" style="float:right; font-size:smaller"}

### DataBlock

>      DataBlock (blocks:list=None, dl_type:TfmdDL=None, getters:list=None,
>                 n_inp:int=None, item_tfms:list=None, batch_tfms:list=None,
>                 get_items=None, splitter=None, get_y=None, get_x=None)

Generic container to quickly build `Datasets` and `DataLoaders`.

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| blocks | list | None | One or more `TransformBlock`s |
| dl_type | TfmdDL | None | Task specific `TfmdDL`, defaults to `block`'s dl_type or`TfmdDL` |
| getters | list | None | Getter functions applied to results of `get_items` |
| n_inp | int | None | Number of inputs |
| item_tfms | list | None | `ItemTransform`s, applied on an item |
| batch_tfms | list | None | `Transform`s or `RandTransform`s, applied by batch |
| get_items | NoneType | None |  |
| splitter | NoneType | None |  |
| get_y | NoneType | None |  |
| get_x | NoneType | None |  |

In [ ]:
show_doc(DataLoaders)


---

[source](https://github.com/fastai/fastai/blob/master/fastai/data/core.py#LNone){target="_blank" style="float:right; font-size:smaller"}

### DataLoaders

>      DataLoaders (*loaders, path:str|pathlib.Path='.', device=None)

Basic wrapper around several `DataLoader`s.

In [ ]:
show_doc(Learner)


---

[source](https://github.com/fastai/fastai/blob/master/fastai/learner.py#LNone){target="_blank" style="float:right; font-size:smaller"}

### Learner



Group together a `model`, some `dls` and a `loss_func` to handle training

In [ ]:
show_doc(ShowGraphCallback)


---

[source](https://github.com/fastai/fastai/blob/master/fastai/callback/progress.py#LNone){target="_blank" style="float:right; font-size:smaller"}

### ShowGraphCallback

>      ShowGraphCallback (after_create=None, before_fit=None, before_epoch=None,
>                         before_train=None, before_batch=None, after_pred=None,
>                         after_loss=None, before_backward=None,
>                         after_cancel_backward=None, after_backward=None,
>                         before_step=None, after_cancel_step=None,
>                         after_step=None, after_cancel_batch=None,
>                         after_batch=None, after_cancel_train=None,
>                         after_train=None, before_validate=None,
>                         after_cancel_validate=None, after_validate=None,
>                         after_cancel_epoch=None, after_epoch=None,
>                         after_cancel_fit=None, after_fit=None)

Update a graph of training and validation loss

In [ ]:
show_doc(CSVLogger)

---

[source](https://github.com/fastai/fastai/blob/master/fastai/callback/progress.py#LNone){target="_blank" style="float:right; font-size:smaller"}

### CSVLogger

>      CSVLogger (fname='history.csv', append=False)

Log the results displayed in `learn.path/fname`

In [ ]:
show_doc(cells3d)

Unknown section Notes


---

### cells3d

>      cells3d ()

3D fluorescence microscopy image of cells.

The returned data is a 3D multichannel array with dimensions provided in
``(z, c, y, x)`` order. Each voxel has a size of ``(0.29 0.26 0.26)``
micrometer. Channel 0 contains cell membranes, channel 1 contains nuclei.

## Engine

Core engine for model training. See tutorials for usage examples.

In [ ]:
#| export
class fastTrainer(Learner):
    """
    A custom implementation of the FastAI Learner class for training models in bioinformatics applications.

    """
    
    def __init__(self, 
                 dataloaders: DataLoaders, # The DataLoader objects containing training and validation datasets.
                 model: callable, # A callable model that will be trained on the dataset.
                 loss_fn: Any | None = None, # The loss function to optimize during training. If None, defaults to a suitable default.
                 optimizer: Optimizer | OptimWrapper = Adam, # The optimizer function to use. Defaults to Adam if not specified.
                 lr: float | slice = 1e-3, # Learning rate for the optimizer. Can be a float or a slice object for learning rate scheduling.
                 splitter: callable = trainable_params, # 
                 callbacks: Callback | MutableSequence | None = None, # A callable that determines which parameters of the model should be updated during training.
                 metrics: Any | MutableSequence | None = None, # Optional list of callback functions to customize training behavior.
                 csv_log: bool = False, # Metrics to evaluate the performance of the model during training.
                 show_graph: bool = True, # Whether to log training history to a CSV file. If True, logs will be appended to 'history.csv'.
                 show_summary: bool = False, # The base directory where models are saved or loaded from. Defaults to None.
                 find_lr: bool = False, # Subdirectory within the base path where trained models are stored. Default is 'models'.
                 find_lr_fn = valley, # Weight decay factor for optimization. Defaults to None.
                 path: str | Path | None = None, # Whether to apply weight decay to batch normalization and bias parameters.
                 model_dir: str | Path = 'models', # Whether to update the batch normalization statistics during training.
                 wd: float | int | None = None, 
                 wd_bn_bias: bool = False, 
                 train_bn: bool = True, 
                 moms: tuple = ..., # Tuple of tuples representing the momentum values for different layers in the model. Defaults to FastAI's default settings if not specified.
                 default_cbs: bool = True, # Automatically include default callbacks such as ShowGraphCallback and CSVLogger.
                 ):
        cbs = callbacks if callbacks is not None else []  # Ensure cbs is a list
        if default_cbs:
            if show_graph:
                cbs.append(ShowGraphCallback())
            if csv_log:
                cbs.append(CSVLogger(fname='history.csv', append=False))
        
        super().__init__(dataloaders, model, loss_fn, optimizer, lr, splitter, cbs, metrics, path, model_dir, wd, wd_bn_bias, train_bn, moms)
        
        if show_summary:
                print(self.summary())
        if find_lr:
                self.lr_find(suggest_funcs=find_lr_fn)
                lr = float('%.1g'%(lr))
                print('Inferred learning rate: ', lr)


## Utils


Utility functions.

In [ ]:
#| export
# maybe this one should be changed for fastai store_attr()
def attributesFromDict(d):
    self = d.pop('self')
    for n, v in d.items():
        setattr(self, n, v)

In [ ]:
#| export
def get_device():
    return torch_device("cuda" if is_cuda_available() else "cpu")

In [ ]:
#| export
def img2float(image, force_copy=False):
    return util.img_as_float(image, force_copy=force_copy)

In [ ]:
#| export
def img2Tensor(image):
    return torchTensor(img2float(image))

---

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()